# 02 — Data Cleaning and Preparation

The initial data check identified several issues that need to be handled before analysis.

In this notebook, the dataset is prepared by:

- Removing duplicate records
- Standardizing categorical values
- Handling missing values
- Converting date fields
- Handling invalid dates
- Parsing nested JSON fields
- Extracting useful patent attributes
- Creating analytical variables
- Inspecting extreme values

The cleaned dataset produced here will be used for exploratory data analysis.

In [74]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import re

ROOT = Path.cwd().parent

RAW_DIR = ROOT / "Data" / "raw"
PROCESSED_DIR = ROOT / "Data" / "processed"
TABLES_DIR = ROOT / "Outputs" / "tables"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = RAW_DIR / "Combined_data.parquet"

print("Dataset:", DATA_PATH)
print("File exists:", DATA_PATH.exists())

Dataset: /Users/janakdobariya/Bramha/NLP_Engineering/BA/ai_patent_business_analytics/Data/raw/Combined_data.parquet
File exists: True


In [75]:
df = pd.read_parquet(DATA_PATH)

print("Dataset loaded.")
print("Rows:", f"{len(df):,}")
print("Columns:", df.shape[1])

Dataset loaded.
Rows: 80,816
Columns: 22


In [76]:
clean_df = df.copy()

print("Working copy created.")

Working copy created.


In [77]:
print(
    "Duplicate publication numbers before cleaning:",
    clean_df["publication_number"].duplicated().sum()
)

Duplicate publication numbers before cleaning: 250


In [78]:
before = len(clean_df)

clean_df = clean_df.drop_duplicates(
    subset="publication_number",
    keep="first"
).copy()

after = len(clean_df)

print("Rows before:", f"{before:,}")
print("Rows after:", f"{after:,}")
print("Rows removed:", f"{before - after:,}")

Rows before: 80,816
Rows after: 80,566
Rows removed: 250


In [79]:
clean_df["country_code"].value_counts(dropna=False)

country_code
US               79872
  US               396
UNKNOWN             57
USA                 55
us                  53
US                  53
U.S.                44
United States       36
Name: count, dtype: int64

In [80]:
clean_df["country_code"] = (
    clean_df["country_code"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [81]:
country_mapping = {
    "USA": "US",
    "U.S.": "US",
    "UNITED STATES": "US"
}

clean_df["country_code"] = (
    clean_df["country_code"]
    .replace(country_mapping)
)

In [82]:
clean_df["country_code"].value_counts(dropna=False)

country_code
US         80509
UNKNOWN       57
Name: count, dtype: int64[pyarrow]

In [83]:
clean_df["kind_code"] = (
    clean_df["kind_code"]
    .astype("string")
    .str.strip()
    .str.upper()
)

clean_df["kind_code"].value_counts(dropna=False)

kind_code
B2    72055
B1     8511
Name: count, dtype: int64[pyarrow]

In [84]:
date_columns = [
    "publication_date",
    "filing_date",
    "grant_date",
    "priority_date"
]

In [85]:
for col in date_columns:

    numeric = pd.to_numeric(
        clean_df[col],
        errors="coerce"
    )

    clean_df[col] = pd.to_datetime(
        numeric.astype("Int64").astype(str),
        format="%Y%m%d",
        errors="coerce"
    )

In [86]:
clean_df[date_columns].head()

,publication_date,filing_date,grant_date,priority_date
0,2021-01-05,2017-06-02,2021-01-05,2016-06-03
1,2021-01-05,2017-01-06,2021-01-05,2012-02-27
2,2021-01-05,2017-08-30,2021-01-05,2017-08-30
3,2021-01-05,2018-09-13,2021-01-05,2018-09-13
4,2021-01-05,2018-07-30,2021-01-05,2017-07-28


In [87]:
date_quality = pd.DataFrame({
    "missing_after_conversion":
        clean_df[date_columns].isna().sum(),

    "missing_percent":
        (
            clean_df[date_columns].isna().mean()
            * 100
        ).round(2)
})

date_quality

,missing_after_conversion,missing_percent
publication_date,0,0.00
filing_date,734,0.91
grant_date,0,0.00
priority_date,1380,1.71


In [88]:
invalid_order = (
    clean_df["filing_date"].notna()
    & clean_df["grant_date"].notna()
    & (
        clean_df["filing_date"]
        > clean_df["grant_date"]
    )
)

print(
    "Filing date after grant date:",
    invalid_order.sum()
)

Filing date after grant date: 41


In [89]:
clean_df["grant_year"] = (
    clean_df["grant_date"].dt.year
)

clean_df["grant_year"].value_counts().sort_index()

grant_year
2021    11246
2022    15219
2023    17597
2024    16828
2025    15294
2026     4382
Name: count, dtype: int64

In [90]:
def safe_json_load(value):
    """
    Convert a JSON string to a Python object.
    Missing or invalid values return an empty list.
    """
    if pd.isna(value):
        return []

    try:
        result = json.loads(value)
        return result if isinstance(result, list) else []
    except (json.JSONDecodeError, TypeError):
        return []

In [91]:
safe_json_load(clean_df["title_raw"].iloc[0])

[{'text': 'Method and system for estimation of stress of a person using photoplethysmography',
  'language': 'en',
  'truncated': False}]

In [92]:
def extract_text(value, preferred_language="en"):
    items = safe_json_load(value)

    # First preference: English
    for item in items:
        if (
            isinstance(item, dict)
            and str(item.get("language", "")).lower() == preferred_language
            and item.get("text")
        ):
            return item["text"].strip()

    # Fallback: first available text
    for item in items:
        if isinstance(item, dict) and item.get("text"):
            return item["text"].strip()

    return np.nan

In [93]:
clean_df["title"] = clean_df["title_raw"].apply(extract_text)

clean_df[
    ["publication_number", "title"]
].head()

,publication_number,title
0,US-10881345-B2,Method and system for estimation of stress of ...
1,US-10881348-B2,System and method for gathering and analyzing ...
2,US-10881463-B2,Optimizing patient treatment recommendations u...
3,US-10881964-B1,Automated detection of emergent behaviors in i...
4,US-10882488-B2,Hardware and software mechanisms on autonomous...


In [94]:
print(
    "Missing titles:",
    clean_df["title"].isna().sum()
)

Missing titles: 241


In [95]:
clean_df["abstract"] = (
    clean_df["abstract_raw"]
    .apply(extract_text)
)

clean_df[
    ["publication_number", "abstract"]
].head()

,publication_number,abstract
0,US-10881345-B2,A system and method for determining a stress l...
1,US-10881348-B2,Systems and methods for measuring biologically...
2,US-10881463-B2,Patient treatment may be optimized using Recur...
3,US-10881964-B1,Various aspects of the subject technology rela...
4,US-10882488-B2,An autonomous robot vehicle includes a front s...


In [96]:
print(
    "Missing abstracts:",
    clean_df["abstract"].isna().sum()
)

Missing abstracts: 725


In [97]:
def extract_text_features(value):
    text = extract_text(value)

    if pd.isna(text):
        return pd.Series({
            "claims_text_length": np.nan,
            "claims_word_count": np.nan
        })

    return pd.Series({
        "claims_text_length": len(text),
        "claims_word_count": len(
            re.findall(r"\b\w+\b", text)
        )
    })

In [98]:
claims_features = (
    clean_df["claims_raw"]
    .apply(extract_text_features)
)

clean_df[
    [
        "claims_text_length",
        "claims_word_count"
    ]
] = claims_features

In [99]:
clean_df[
    [
        "claims_text_length",
        "claims_word_count"
    ]
].describe()

,claims_text_length,claims_word_count
count,8.056600e+04,80566.000000
mean,1.009160e+04,1469.244483
std,1.241274e+04,1033.006871
min,5.570000e+02,86.000000
25%,7.010250e+03,1045.000000
50%,9.051000e+03,1354.000000
75%,1.169200e+04,1747.000000
max,2.605094e+06,182826.000000


In [100]:
def word_count(value):
    if pd.isna(value):
        return np.nan

    return len(
        re.findall(
            r"\b\w+\b",
            str(value)
        )
    )

In [101]:
clean_df["title_word_count"] = (
    clean_df["title"]
    .apply(word_count)
)

clean_df["abstract_word_count"] = (
    clean_df["abstract"]
    .apply(word_count)
)

In [102]:
clean_df[
    [
        "title_word_count",
        "abstract_word_count",
        "claims_word_count"
    ]
].describe()

,title_word_count,abstract_word_count,claims_word_count
count,80325.000000,79841.000000,80566.000000
mean,9.461276,121.140893,1469.244483
std,4.299415,32.893476,1033.006871
min,1.000000,4.000000,86.000000
25%,6.000000,99.000000,1045.000000
50%,9.000000,128.000000,1354.000000
75%,12.000000,147.000000,1747.000000
max,62.000000,533.000000,182826.000000


In [103]:
def extract_names(value):
    items = safe_json_load(value)

    names = []

    for item in items:
        if isinstance(item, dict):
            name = item.get("name")

            if name:
                name = str(name).strip()

                if name and name not in names:
                    names.append(name)

    return names

In [104]:
clean_df["inventor_names"] = (
    clean_df["inventors_raw"]
    .apply(extract_names)
)

clean_df["inventor_count"] = (
    clean_df["inventor_names"]
    .apply(len)
)

In [105]:
clean_df[
    [
        "publication_number",
        "inventor_names",
        "inventor_count"
    ]
].head()

,publication_number,inventor_names,inventor_count
0,US-10881345-B2,"[UNNI MIDHUN PARAKKAL, JAYARAMAN SRINIVASAN, P...",3
1,US-10881348-B2,"[LEVINE BRIAN, MARCI CARL, KOTHURI RAVI KANTH V]",3
2,US-10881463-B2,"[MEI JING, ZHAO SHI WAN, HU GANG, LI JING, XIA...",6
3,US-10881964-B1,[DILLS THOMAS BRADLEY],1
4,US-10882488-B2,"[FERGUSON DAVID, ZHU JIAJUN, VINES NICK, SMITH...",4


In [106]:
clean_df["inventor_count"].describe()

count    80566.000000
mean         3.540042
std          2.350370
min          0.000000
25%          2.000000
50%          3.000000
75%          5.000000
max         36.000000
Name: inventor_count, dtype: float64

In [107]:
clean_df["assignee_names"] = (
    clean_df["assignees_raw"]
    .apply(extract_names)
)

clean_df["assignee_count"] = (
    clean_df["assignee_names"]
    .apply(len)
)

In [108]:
def extract_primary_assignee(value):
    items = safe_json_load(value)

    for item in items:
        if isinstance(item, dict):
            name = item.get("name")

            if name:
                return str(name).strip()

    return np.nan

In [109]:
def extract_assignee_country(value):
    items = safe_json_load(value)

    for item in items:
        if isinstance(item, dict):
            country = item.get("country_code")

            if country:
                return str(country).strip().upper()

    return np.nan

In [110]:
clean_df["primary_assignee"] = (
    clean_df["assignees_raw"]
    .apply(extract_primary_assignee)
)

clean_df["assignee_country"] = (
    clean_df["assignees_raw"]
    .apply(extract_assignee_country)
)

In [111]:
clean_df[
    [
        "primary_assignee",
        "assignee_country",
        "assignee_count"
    ]
].head(10)

,primary_assignee,assignee_country,assignee_count
0,TATA CONSULTANCY SERVICES LTD,IN,1
1,NIELSEN CO US LLC,US,1
2,IBM,US,1
3,ELECTRONIC ARTS INC,US,1
4,NURO INC,US,1
5,TOYOTA RES INST INC,US,1
6,EMERALD THERAPEUTICS INC,US,1
7,VESTAS WIND SYS AS,DK,1
8,BEIJING DIDI INFINITY TECHNOLOGY & DEV CO LTD,CN,1
9,WAYMO LLC,US,1


In [112]:
clean_df["primary_assignee"] = (
    clean_df["primary_assignee"]
    .fillna("Unknown")
)

clean_df["assignee_country"] = (
    clean_df["assignee_country"]
    .fillna("Unknown")
)

In [113]:
clean_df["assignee_country"].value_counts().head(15)

assignee_country
US         50940
KR          5176
CN          5145
JP          4634
DE          2180
Unknown     1699
CA          1329
GB          1293
IL          1062
TW           871
IE           762
IN           662
FR           575
CH           568
NL           544
Name: count, dtype: int64

In [114]:
def extract_cpc_codes(value):
    items = safe_json_load(value)

    codes = []

    for item in items:
        if isinstance(item, dict):
            code = item.get("code")

            if code:
                code = (
                    str(code)
                    .strip()
                    .upper()
                    .replace(" ", "")
                )

                if code not in codes:
                    codes.append(code)

    return codes

In [115]:
clean_df["cpc_codes"] = (
    clean_df["cpc_raw"]
    .apply(extract_cpc_codes)
)

clean_df["cpc_count"] = (
    clean_df["cpc_codes"]
    .apply(len)
)

In [116]:
clean_df[
    [
        "publication_number",
        "cpc_codes",
        "cpc_count"
    ]
].head()

,publication_number,cpc_codes,cpc_count
0,US-10881345-B2,"[G06N3/0499, G06N3/09, G16Z99/00, A61B5/02416,...",17
1,US-10881348-B2,"[G06Q10/40, G09B25/00, G06N20/00, A61B5/0816, ...",18
2,US-10881463-B2,"[G06N3/044, G06N3/092, G06N3/0442, G06N3/08, G...",13
3,US-10881964-B1,"[A63F13/75, A63F13/67, G06V10/82, G06V10/764, ...",17
4,US-10882488-B2,"[B64U2201/10, G05D1/651, G05D1/2235, G05D1/644...",110


In [117]:
def primary_ai_cpc(codes):
    for code in codes:
        if code.startswith("G06N"):
            return code

    return np.nan

In [118]:
clean_df["primary_ai_cpc"] = (
    clean_df["cpc_codes"]
    .apply(primary_ai_cpc)
)

print(
    "Patents without G06N after parsing:",
    clean_df["primary_ai_cpc"].isna().sum()
)

Patents without G06N after parsing: 0


In [119]:
def extract_cpc_group(code):
    if pd.isna(code):
        return np.nan

    match = re.match(
        r"^(G06N\d+)",
        str(code)
    )

    if match:
        return match.group(1)

    return np.nan

In [120]:
clean_df["cpc_group"] = (
    clean_df["primary_ai_cpc"]
    .apply(extract_cpc_group)
)

clean_df["cpc_group"].value_counts().head(15)

cpc_group
G06N3     41981
G06N20    22218
G06N5      9418
G06N7      4091
G06N10     2783
G06N99       75
Name: count, dtype: int64

In [121]:
def backward_citation_count(value):
    items = safe_json_load(value)

    publication_numbers = set()

    for item in items:
        if isinstance(item, dict):
            number = item.get("publication_number")

            if number:
                publication_numbers.add(
                    str(number).strip()
                )

    return len(publication_numbers)

In [122]:
clean_df["backward_citation_count"] = (
    clean_df["citations_raw"]
    .apply(backward_citation_count)
)

In [123]:
clean_df[
    "backward_citation_count"
].describe()

count    80566.000000
mean        37.143969
std        128.831438
min          0.000000
25%          7.000000
50%         13.000000
75%         25.000000
max       7566.000000
Name: backward_citation_count, dtype: float64

In [53]:
clean_df["filing_to_grant_days"] = (
    clean_df["grant_date"]
    - clean_df["filing_date"]
).dt.days

In [54]:
clean_df[
    "filing_to_grant_days"
].describe()

count    79832.000000
mean      1041.039934
std       1078.828791
min     -10903.000000
25%        688.000000
50%        985.000000
75%       1314.000000
max      46124.000000
Name: filing_to_grant_days, dtype: float64

In [55]:
print(
    "Negative filing-to-grant values:",
    (
        clean_df["filing_to_grant_days"] < 0
    ).sum()
)

Negative filing-to-grant values: 41


In [56]:
analysis_columns = [
    "publication_number",
    "application_number",
    "family_id",
    "filing_date",
    "grant_date",
    "grant_year",
    "title",
    "abstract",
    "inventor_count",
    "assignee_count",
    "primary_assignee",
    "assignee_country",
    "cpc_count",
    "primary_ai_cpc",
    "cpc_group",
    "backward_citation_count",
    "title_word_count",
    "abstract_word_count",
    "claims_text_length",
    "claims_word_count",
    "filing_to_grant_days"
]

clean_df[
    analysis_columns
].head()

,publication_number,application_number,family_id,filing_date,grant_date,grant_year,title,abstract,inventor_count,assignee_count,...,assignee_country,cpc_count,primary_ai_cpc,cpc_group,backward_citation_count,title_word_count,abstract_word_count,claims_text_length,claims_word_count,filing_to_grant_days
0,US-10881345-B2,US-201715611953-A,59030779,2017-06-02,2021-01-05,2021,Method and system for estimation of stress of ...,A system and method for determining a stress l...,3,1,...,IN,17,G06N3/0499,G06N3,8,12.0,136.0,8263,1277,1313.0
1,US-10881348-B2,US-201715400287-A,49380436,2017-01-06,2021-01-05,2021,System and method for gathering and analyzing ...,Systems and methods for measuring biologically...,3,1,...,US,18,G06N20/00,G06N20,744,18.0,145.0,9630,1497,1460.0
2,US-10881463-B2,US-201715690436-A,65436394,2017-08-30,2021-01-05,2021,Optimizing patient treatment recommendations u...,Patient treatment may be optimized using Recur...,6,1,...,US,13,G06N3/044,G06N3,22,15.0,125.0,11398,1686,1224.0
3,US-10881964-B1,US-201816130854-A,74045007,2018-09-13,2021-01-05,2021,Automated detection of emergent behaviors in i...,Various aspects of the subject technology rela...,1,1,...,US,17,G06N3/045,G06N3,5,12.0,155.0,9597,1393,845.0
4,US-10882488-B2,US-201816048797-A,63207881,2018-07-30,2021-01-05,2021,Hardware and software mechanisms on autonomous...,An autonomous robot vehicle includes a front s...,4,1,...,US,110,G06N20/00,G06N20,21,10.0,100.0,6702,993,890.0


In [57]:
important_columns = [
    "publication_number",
    "filing_date",
    "grant_date",
    "title",
    "abstract",
    "primary_assignee",
    "assignee_country",
    "inventor_count",
    "assignee_count",
    "cpc_count",
    "primary_ai_cpc",
    "cpc_group",
    "backward_citation_count",
    "claims_text_length",
    "filing_to_grant_days"
]

missing_after_cleaning = pd.DataFrame({
    "missing_count": clean_df[important_columns].isna().sum(),
    "missing_percent": (
        clean_df[important_columns].isna().mean() * 100
    ).round(2)
})

missing_after_cleaning = (
    missing_after_cleaning
    .sort_values("missing_count", ascending=False)
)

missing_after_cleaning

,missing_count,missing_percent
filing_date,734,0.91
filing_to_grant_days,734,0.91
abstract,725,0.90
title,241,0.30
publication_number,0,0.00
grant_date,0,0.00
primary_assignee,0,0.00
assignee_country,0,0.00
inventor_count,0,0.00
assignee_count,0,0.00


In [58]:
print(
    "PCT number missing percentage:",
    round(clean_df["pct_number"].isna().mean() * 100, 2),
    "%"
)

PCT number missing percentage: 90.35 %


In [59]:
invalid_duration = (
    clean_df["filing_to_grant_days"] < 0
)

print(
    "Negative filing-to-grant values:",
    invalid_duration.sum()
)

Negative filing-to-grant values: 41


In [60]:
clean_df.loc[
    clean_df["filing_to_grant_days"] < 0,
    "filing_to_grant_days"
] = np.nan

In [61]:
print(
    "Negative values remaining:",
    (clean_df["filing_to_grant_days"] < 0).sum()
)

Negative values remaining: 0


In [62]:
print("Earliest grant:", clean_df["grant_date"].min())
print("Latest grant:", clean_df["grant_date"].max())

Earliest grant: 2021-01-05 00:00:00
Latest grant: 2026-04-21 00:00:00


In [63]:
outside_scope = (
    (clean_df["grant_date"] < "2021-01-01")
    |
    (clean_df["grant_date"] > "2026-06-30")
)

print(
    "Grant dates outside study period:",
    outside_scope.sum()
)

Grant dates outside study period: 0


In [64]:
numeric_columns = [
    "inventor_count",
    "assignee_count",
    "cpc_count",
    "backward_citation_count",
    "title_word_count",
    "abstract_word_count",
    "claims_text_length",
    "claims_word_count",
    "filing_to_grant_days"
]

clean_df[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
inventor_count,80566.0,3.540042,2.350370,0.0,2.00,3.0,5.0,36.0
assignee_count,80566.0,1.076620,0.459061,0.0,1.00,1.0,1.0,24.0
cpc_count,80566.0,13.298885,11.853232,1.0,7.00,11.0,16.0,147.0
backward_citation_count,80566.0,37.143969,128.831438,0.0,7.00,13.0,25.0,7566.0
title_word_count,80325.0,9.461276,4.299415,1.0,6.00,9.0,12.0,62.0
abstract_word_count,79841.0,121.140893,32.893476,4.0,99.00,128.0,147.0,533.0
claims_text_length,80566.0,10091.596641,12412.743191,557.0,7010.25,9051.0,11692.0,2605094.0
claims_word_count,80566.0,1469.244483,1033.006871,86.0,1045.00,1354.0,1747.0,182826.0
filing_to_grant_days,79791.0,1046.658896,1050.189777,70.0,690.00,985.0,1314.0,46124.0


In [65]:
outlier_summary = []

for col in numeric_columns:

    series = clean_df[col].dropna()

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_count = (
        (series < lower) |
        (series > upper)
    ).sum()

    outlier_summary.append({
        "variable": col,
        "Q1": q1,
        "Q3": q3,
        "lower_bound": lower,
        "upper_bound": upper,
        "potential_extreme_values": outlier_count,
        "percent": round(
            outlier_count / len(series) * 100,
            2
        )
    })

outlier_summary = pd.DataFrame(outlier_summary)

outlier_summary

,variable,Q1,Q3,lower_bound,upper_bound,potential_extreme_values,percent
0,inventor_count,2.00,5.0,-2.500,9.500,1840,2.28
1,assignee_count,1.00,1.0,1.000,1.000,7475,9.28
2,cpc_count,7.00,16.0,-6.500,29.500,3630,4.51
3,backward_citation_count,7.00,25.0,-20.000,52.000,9025,11.20
4,title_word_count,6.00,12.0,-3.000,21.000,1136,1.41
5,abstract_word_count,99.00,147.0,27.000,219.000,685,0.86
6,claims_text_length,7010.25,11692.0,-12.375,18714.625,3406,4.23
7,claims_word_count,1045.00,1747.0,-8.000,2800.000,2780,3.45
8,filing_to_grant_days,690.00,1314.0,-246.000,2250.000,1041,1.30


In [66]:
clean_df[
    [
        "publication_number",
        "title",
        "claims_text_length"
    ]
].sort_values(
    "claims_text_length",
    ascending=False
).head(10)

,publication_number,title,claims_text_length
73761,US-12254961-B2,Hierarchical machine learning techniques for i...,2605094
72333,US-12381769-B2,AI/ML empowered high order modulation,1220759
6310,US-11080336-B2,"System and method for fuzzy concept mapping, v...",1161183
70383,US-12223404-B2,Iterative attention-based neural network train...,279834
63265,US-12327166-B1,Iterative attention-based neural network train...,263201
51159,US-12118096-B2,Image encryption method based on multi-scale c...,174977
29596,US-11540781-B2,Modeling a neuronal controller exhibiting huma...,173744
29978,US-11546086-B2,Channel decoding method and channel decoding d...,166330
19751,US-11521102-B2,"Transformation apparatus, decision apparatus, ...",162704
15772,US-11423201-B2,Method and system for determining helicopter r...,162159


In [67]:
clean_df[
    [
        "publication_number",
        "title",
        "backward_citation_count"
    ]
].sort_values(
    "backward_citation_count",
    ascending=False
).head(10)

,publication_number,title,backward_citation_count
15946,US-11423886-B2,Task flow identification based on user intent,7566
18469,US-11500672-B2,Distributed personal assistant,7345
80392,US-12605104-B2,Method and apparatus for neuroenhancement,6829
34718,US-11723579-B2,Method and apparatus for neuroenhancement,6826
4002,US-11010550-B2,Unified language modeling framework for word p...,4811
62052,US-12431128-B2,Task flow identification based on user intent,4195
57796,US-12165635-B2,Intelligent automated assistant,4000
66402,US-12204932-B2,Distributed personal assistant,3994
47661,US-12001933-B2,Virtual assistant in a communication session,3965
55523,US-12154016-B2,Virtual assistant in a communication session,3965


In [68]:
clean_df[
    [
        "publication_number",
        "title",
        "cpc_count"
    ]
].sort_values(
    "cpc_count",
    ascending=False
).head(10)

,publication_number,title,cpc_count
3903,US-11005720-B2,System and method for a vehicle zone-determine...,147
35228,US-11571263-B2,Mixed-reality surgical system with physical ma...,142
17301,US-11478310-B2,Virtual guidance for ankle surgery procedures,142
27071,US-11645531-B2,Mixed-reality surgical system with physical ma...,142
27749,US-11657287-B2,Virtual guidance for ankle surgery procedures,142
16711,US-11439469-B2,Virtual guidance for orthopedic surgical proce...,142
51654,US-12125577-B2,Mixed reality-aided education using virtual mo...,141
53335,US-12050999-B2,Virtual guidance for orthopedic surgical proce...,141
7990,US-11126825-B2,Natural language interaction for smart assistant,141
53144,US-12046349-B2,Visualization of intraoperatively modified sur...,141


In [69]:
final_columns = [
    "publication_number",
    "application_number",
    "family_id",

    "publication_date",
    "filing_date",
    "grant_date",
    "priority_date",
    "grant_year",

    "title",
    "abstract",

    "inventor_count",
    "assignee_count",

    "primary_assignee",
    "assignee_country",

    "cpc_count",
    "primary_ai_cpc",
    "cpc_group",

    "backward_citation_count",

    "title_word_count",
    "abstract_word_count",
    "claims_text_length",
    "claims_word_count",

    "filing_to_grant_days"
]

analysis_df = clean_df[final_columns].copy()

print("Final analytical dataset shape:")
print(analysis_df.shape)

analysis_df.head()

Final analytical dataset shape:
(80566, 23)


,publication_number,application_number,family_id,publication_date,filing_date,grant_date,priority_date,grant_year,title,abstract,...,assignee_country,cpc_count,primary_ai_cpc,cpc_group,backward_citation_count,title_word_count,abstract_word_count,claims_text_length,claims_word_count,filing_to_grant_days
0,US-10881345-B2,US-201715611953-A,59030779,2021-01-05,2017-06-02,2021-01-05,2016-06-03,2021,Method and system for estimation of stress of ...,A system and method for determining a stress l...,...,IN,17,G06N3/0499,G06N3,8,12.0,136.0,8263,1277,1313.0
1,US-10881348-B2,US-201715400287-A,49380436,2021-01-05,2017-01-06,2021-01-05,2012-02-27,2021,System and method for gathering and analyzing ...,Systems and methods for measuring biologically...,...,US,18,G06N20/00,G06N20,744,18.0,145.0,9630,1497,1460.0
2,US-10881463-B2,US-201715690436-A,65436394,2021-01-05,2017-08-30,2021-01-05,2017-08-30,2021,Optimizing patient treatment recommendations u...,Patient treatment may be optimized using Recur...,...,US,13,G06N3/044,G06N3,22,15.0,125.0,11398,1686,1224.0
3,US-10881964-B1,US-201816130854-A,74045007,2021-01-05,2018-09-13,2021-01-05,2018-09-13,2021,Automated detection of emergent behaviors in i...,Various aspects of the subject technology rela...,...,US,17,G06N3/045,G06N3,5,12.0,155.0,9597,1393,845.0
4,US-10882488-B2,US-201816048797-A,63207881,2021-01-05,2018-07-30,2021-01-05,2017-07-28,2021,Hardware and software mechanisms on autonomous...,An autonomous robot vehicle includes a front s...,...,US,110,G06N20/00,G06N20,21,10.0,100.0,6702,993,890.0


In [70]:
print("=" * 50)
print("FINAL DATA CHECK")
print("=" * 50)

print("Rows:", f"{len(analysis_df):,}")
print("Columns:", analysis_df.shape[1])

print(
    "Duplicate publication numbers:",
    analysis_df["publication_number"]
    .duplicated()
    .sum()
)

print(
    "Missing grant dates:",
    analysis_df["grant_date"]
    .isna()
    .sum()
)

print(
    "Invalid grant years:",
    (~analysis_df["grant_year"]
      .between(2021, 2026))
    .sum()
)

print(
    "Missing CPC group:",
    analysis_df["cpc_group"]
    .isna()
    .sum()
)

FINAL DATA CHECK
Rows: 80,566
Columns: 23
Duplicate publication numbers: 0
Missing grant dates: 0
Invalid grant years: 0
Missing CPC group: 0


In [71]:
year_check = (
    analysis_df
    .groupby("grant_year")
    .size()
    .reset_index(name="patent_count")
)

year_check

,grant_year,patent_count
0,2021,11246
1,2022,15219
2,2023,17597
3,2024,16828
4,2025,15294
5,2026,4382


In [ ]:
CLEAN_PATH = (
    PROCESSED_DIR /
    "02_clean_patents.parquet"
)

analysis_df.to_parquet(
    CLEAN_PATH,
    index=False
)

print("Clean dataset saved:")
print(CLEAN_PATH)

In [ ]:
CSV_PATH = (
    PROCESSED_DIR /
    "02_clean_patents.csv"
)

analysis_df.to_csv(
    CSV_PATH,
    index=False
)

print("CSV saved:")
print(CSV_PATH)

## Cleaning Summary

The dataset is now ready for analysis.

The preparation process included:

- Removal of duplicate patent records
- Standardization of categorical values
- Conversion and validation of date fields
- Parsing of nested patent information
- Extraction of inventor, assignee, CPC and citation information
- Creation of analytical variables
- Review of missing and extreme values

The cleaned dataset will be used in the exploratory data analysis.